# Phase-Space Coordinates and Wigner Conventions


This companion notebook explains a common source of confusion in bosonic tutorials: the same cavity state can be plotted either in quadrature coordinates `(x, p)` or in coherent-state coordinates `alpha`.

For a coherent state centered at $\alpha = 2$, the Wigner peak sits near $x = \sqrt{2} \alpha \approx 2.83$ in quadrature coordinates, but near $\alpha = 2$ when the plotting routine is asked to use coherent-state coordinates directly. The underlying state is the same; only the axis convention changes.


## Imports


In [ ]:

from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from tutorials.workflow_tutorial_support import configure_notebook_style

configure_notebook_style()

from cqed_sim import cavity_wigner

## Physics / model definition


In [ ]:
alpha = 2.0 + 0.0j
n_cav = 30
rho_cavity = qt.coherent_dm(n_cav, alpha)
expected_quadrature_center = np.sqrt(2.0) * np.real(alpha)
expected_alpha_center = np.real(alpha)

## Pulse / sequence construction


In [ ]:
print("No control pulse is needed here: we are only changing the plotting convention for the same prepared coherent state.")


## Simulation


In [ ]:
x_quad, y_quad, w_quad = cavity_wigner(rho_cavity, n_points=121, extent=4.6, coordinate="quadrature")
x_alpha, y_alpha, w_alpha = cavity_wigner(rho_cavity, n_points=121, extent=4.6, coordinate="alpha")

quad_index = np.unravel_index(np.argmax(w_quad), w_quad.shape)
alpha_index = np.unravel_index(np.argmax(w_alpha), w_alpha.shape)
quadrature_peak = (float(x_quad[quad_index[1]]), float(y_quad[quad_index[0]]))
alpha_peak = (float(x_alpha[alpha_index[1]]), float(y_alpha[alpha_index[0]]))

print("Expected quadrature-space center:", expected_quadrature_center)
print("Numerical quadrature-space peak:", quadrature_peak)
print("Expected alpha-space center:", expected_alpha_center)
print("Numerical alpha-space peak:", alpha_peak)


## Analysis / visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))
images = []
for axis, xvec, yvec, values, title, xlabel, ylabel, xmark in [
    (axes[0], x_quad, y_quad, w_quad, "Quadrature coordinates", "x", "p", expected_quadrature_center),
    (axes[1], x_alpha, y_alpha, w_alpha, "Coherent-state coordinates", r"Re($\alpha$)", r"Im($\alpha$)", expected_alpha_center),
]:
    image = axis.imshow(
    values,
    origin="lower",
    extent=[xvec[0], xvec[-1], yvec[0], yvec[-1]],
    cmap="RdBu_r",
    aspect="equal",
    )
    images.append(image)
    axis.axvline(xmark, color="#E45756", ls="--", lw=1.0)
    axis.set_title(title)
    axis.set_xlabel(xlabel)
    axis.set_ylabel(ylabel)

fig.colorbar(images[-1], ax=axes.tolist(), shrink=0.86, label="Wigner value")
plt.tight_layout()
plt.show()


## Interpretation


This notebook is a sanity check on plotting conventions, not a different physical preparation. The two panels differ only by the phase-space coordinate system used by the plotting routine.

Use `coordinate="alpha"` when you want the axes to match coherent-state amplitudes directly. Use `coordinate="quadrature"` when you want canonical oscillator axes. Both are valid, but the interpretation of the peak position changes by a factor of $\sqrt{2}$.


## Variations / exercises


- Repeat the comparison for a complex displacement such as `alpha = 1.5 + 0.8j`.
- Overlay the expected centers by hand and confirm the numerical peak follows the same rule.
- Use this notebook as a quick sanity check whenever a bosonic tutorial plot looks shifted.
